# Preprocesamiento

In [1]:
import os, json, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
TARGET = "SalePrice"
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)

## Cargar datos

In [2]:
df = pd.read_csv("data/train.csv")
print("Shape:", df.shape)
print("SalePrice min/mediana/max:", df[TARGET].min(), df[TARGET].median(), df[TARGET].max())
df.head()

Shape: (1168, 81)
SalePrice min/mediana/max: 34900 165000.0 745000


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,255,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2010,WD,Normal,145000
1,1067,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2009,WD,Normal,178000
2,639,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,85000
3,800,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,6,2007,WD,Normal,175000
4,381,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,127000


## Separar X / y

In [3]:
X = df.drop(columns=[TARGET, "Id"])
y = df[TARGET].astype(np.float32)

# Split reproducible. Estratificamos por cuantiles del precio para mantener una
# distribución de precios parecida entre train y validación.
y_bins = pd.qcut(y, q=10, duplicates="drop")
X_train, X_val, y_train_usd, y_val_usd = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y_bins
)
print("Train:", X_train.shape, "Val:", X_val.shape)
print("Mediana train:", float(y_train_usd.median()))
print("Mediana val:", float(y_val_usd.median()))

Train: (934, 79) Val: (234, 79)
Mediana train: 165000.0
Mediana val: 165000.0


## Pipeline de variables

In [4]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, numeric_cols),
    ("cat", cat_pipe, categorical_cols),
], remainder="drop")

X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_processed = preprocessor.transform(X_val).astype(np.float32)

print("Numéricas:", len(numeric_cols))
print("Categóricas:", len(categorical_cols))
print("X train procesado:", X_train_processed.shape)
print("X val procesado:", X_val_processed.shape)
print("NaN train:", np.isnan(X_train_processed).sum())
print("NaN val:", np.isnan(X_val_processed).sum())

Numéricas: 36
Categóricas: 43
X train procesado: (934, 278)
X val procesado: (234, 278)
NaN train: 0
NaN val: 0


## Estandarizar el target en dólares

In [5]:
y_mean = float(y_train_usd.mean())
y_std = float(y_train_usd.std(ddof=0))

y_train_scaled = ((y_train_usd.to_numpy() - y_mean) / y_std).astype(np.float32)
y_val_scaled = ((y_val_usd.to_numpy() - y_mean) / y_std).astype(np.float32)

print(f"Media target train: ${y_mean:,.2f}")
print(f"Std target train:   ${y_std:,.2f}")
print("Target scaled train: media=", y_train_scaled.mean(), "std=", y_train_scaled.std())

Media target train: $181,637.14
Std target train:   $78,015.42
Target scaled train: media= -1.9910759e-07 std= 1.0


In [6]:
np.save("data/X_train_processed.npy", X_train_processed)
np.save("data/X_val_processed.npy", X_val_processed)
np.save("data/y_train_scaled.npy", y_train_scaled)
np.save("data/y_val_scaled.npy", y_val_scaled)
np.save("data/y_train_usd.npy", y_train_usd.to_numpy(dtype=np.float32))
np.save("data/y_val_usd.npy", y_val_usd.to_numpy(dtype=np.float32))
np.save("data/X_train_index.npy", X_train.index.to_numpy())
np.save("data/X_val_index.npy", X_val.index.to_numpy())

joblib.dump(preprocessor, "models/preprocessor_v2.joblib")
joblib.dump({"mean": y_mean, "std": y_std}, "models/target_scaler_v2.joblib")

feature_names = preprocessor.get_feature_names_out().tolist()
with open("models/feature_names_v2.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

print("Guardado correctamente.")
print("Número final de features:", len(feature_names))

Guardado correctamente.
Número final de features: 278


## Verificación final

In [7]:
assert X_train_processed.shape[0] == len(y_train_scaled)
assert X_val_processed.shape[0] == len(y_val_scaled)
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_val_processed).all()
assert np.isfinite(y_train_scaled).all()
assert np.isfinite(y_val_scaled).all()
print("OK: datos listos para baseline y MLP.")

OK: datos listos para baseline y MLP.
